# Multisensory Integration: Audio-Visual

**Author:** Teo Fantacci  
**Purpose:** Pedagogical companion to Froudist-Walsh et al. (in press, *Biol Psychiatry*) — illustrates an audiovisual reaction-time model with three competing architectures (OR / SUM / REPEAT) following Brady & Butler (2025). Corresponds to **Fig 3D** and **§3.3** of the review.

## How to cite
If you find this code useful for your research, please cite:
- The original paper: Brady RM, Butler JS (2025): Modelling Audio-Visual Reaction Time with Recurrent Mean-Field Networks. bioRxiv, p 2025.05.29.656149.
- The review: Froudist-Walsh S, Ivanov TG, Magrou L, Cho H, Busch AN, Muller LE, et al. (in press): Mechanistic computational models of cognition across scales and species. *Biol Psychiatry*.

## Requirements
```bash
pip install numpy matplotlib
```
Runs in Google Colab (recommended) or locally with Jupyter (Python ≥ 3.8).  
No external data files required — all simulations are self-contained.

## Notes
All code is original. No third-party data or figures are included.
Simulation outputs are generated at runtime and not stored in the repository.


## Model overview

This notebook implements the three audio-visual integration architectures of **Brady & Butler (2025)**, *Modelling Audio-Visual Reaction Time with Recurrent Mean-Field Networks* (bioRxiv 2025.05.29.656149).

All three architectures reuse the **Wong & Wang (2006) reduced two-variable decision module** (the same building block as the Winner-Take-All notebook) and compose multiple instances of it to model multisensory tasks.

## The three architectures

- **OR Model** — two independent unisensory modules (audio: `A_H`, `A_M`; visual: `V_H`, `V_M`). On AV trials both modules receive their own stimulus and run in parallel; the decision is the **first** of the four populations to cross threshold (winner-take-all *across* modalities). This produces a small, statistically sub-optimal multisensory benefit.

- **SUM Model** — same two unisensory modules, plus a third "summation" decision module (`SUM_H`, `SUM_M`) that receives input from both upstream modules via a coupling current. The decision is read from the SUM populations, not the unisensory ones. This produces optimal-to-supra-optimal multisensory integration. **Note**: Brady & Butler (2025) do not specify whether the SUM coupling is rate-based (AMPA-style) or NMDA-gating-based, so we expose both options below.

- **REPEAT Model** — same architecture as OR, plus a trial-history modification: on a "repeat" trial (current modality matched the previous one) the cued Hit population is primed. We expose two interpretations of the paper's wording ("baseline activity increase of one standard deviation of the noise"): (i) `IC_bump`, raising the initial S of the cued Hit population, and (ii) `sigma_increase`, multiplying the OU noise σ on the cued Hit population for that trial. A third biologically more plausible implementation is the "active-silent" mechanism, but still done.

## Dynamics

**Notation:** $\alpha \in \{A, V, \text{SUM}\}$ indexes the modules (audio, visual, summation); $m \in \{H, M\}$ indexes the Hit and Miss populations within a module. Each $(\alpha, m)$ pair is a full population — e.g. $(\alpha,m) = (A, H)$ is the audio-Hit pool. Firing rates $r_{\alpha,m}$ are obtained from the W&W f-I curve $H(\cdot)$; $S_{\alpha,m}$ is the slow (NMDA-like) synaptic gating variable; $J$ are weights; $\tau$ are time constants.

Each population has a **single** slow gating variable (W&W reduced 2-variable form — no separate AMPA gating channel):

$$
\frac{dS_{\alpha,m}}{dt} = -\frac{S_{\alpha,m}}{\tau_\alpha} + \gamma\,(1 - S_{\alpha,m})\, H(x_{\alpha,m})
$$

Input currents to the Hit and Miss pools of module $\alpha$ (self-excitation $J_{HH}, J_{MM}$ and cross-inhibition $J_{HM}, J_{MH}$):

$$
x_{\alpha,H} = J_{HH}\, S_{\alpha,H} - J_{HM}\, S_{\alpha,M} + I_0 + I^{\text{ext}}_{\alpha,H}(t) + \eta_{\alpha,H}(t)
$$

$$
x_{\alpha,M} = J_{MM}\, S_{\alpha,M} - J_{MH}\, S_{\alpha,H} + I_0 + I^{\text{ext}}_{\alpha,M}(t) + \eta_{\alpha,M}(t)
$$

W&W reduced f-I curve (firing rate, Hz):

$$
r_{\alpha,m} = H(x_{\alpha,m}) = \frac{a\, x_{\alpha,m} - b}{1 - \exp\bigl(-d\,(a\, x_{\alpha,m} - b)\bigr)}
$$

For the **SUM** module ($\alpha = \text{SUM}$), the input includes a coupling current from both unisensory modules. Two interpretations (the paper does not specify):

$$
I^{\text{couple,AMPA}}_{m} = J^{\text{AMPA}}_{\text{couple}}\,\bigl(r_{A,m} + r_{V,m}\bigr)
\qquad
I^{\text{couple,NMDA}}_{m} = J^{\text{NMDA}}_{\text{couple}}\,\bigl(S_{A,m} + S_{V,m}\bigr)
$$

so the SUM-module current becomes $x_{\text{SUM},m} = J_{mm}\, S_{\text{SUM},m} - J_{m\bar m}\, S_{\text{SUM},\bar m} + I_0 + I^{\text{couple}}_m + \eta_{\text{SUM},m}(t)$, with no direct external stimulus.

A decision occurs the first time any population's rate crosses a threshold (Hit pop → "Hit", Miss pop → "Miss"). In the **OR** model the race is among the 4 unisensory pops; in **SUM** it is among the 2 SUM pops; in **REPEAT** it is among the 4 unisensory pops, but on a repeat trial the cued Hit pool starts with $S_{\alpha,H}(0) = \text{IC}_{\text{bump}}$ (or with an inflated noise σ, depending on `repeat_mode`).

Brady & Butler **reinterpret** $\tau_\alpha$ as a per-participant, per-modality accumulation rate (faster $\tau_\alpha \Rightarrow$ faster decisions), sweeping over $[0.080, 0.130]$ s instead of holding it at the biophysical NMDA value $\tau_S = 0.100$ s.

## Reference and scope

The **single-module dynamics are exactly Wong & Wang (2006)** Eqs. 1–6 (also Brady & Butler Eqs. 1–6): one Hit and one Miss population per area, self-excitation (`J_HH`, `J_MM`), cross-inhibition (`J_HM`, `J_MH`), the reduced f-I curve `H(x) = (a x − b) / (1 − exp(−d (a x − b)))`, slow NMDA gating with `dS/dt = −S/τ + (1−S) γ H(x)`, and OU noise on the input current with `τ_AMPA = 2 ms`.

The **per-modality `τ_S` is reinterpreted by Brady & Butler** as a per-participant accumulation rate (faster `τ_S` → faster decisions). They sweep `τ_A, τ_V ∈ {0.080, …, 0.130} s` over 121 simulated participants. We start with a single representative participant (`τ_A = τ_V = 0.100 s` — the W&W default) and 10 trials per condition, and leave the full sweep + GDDM fitting for later.

## Open assumptions (paper does not fully specify)

1. **Brady & Butler state `J_ext = 5.2 × 10⁻⁵ nA·Hz⁻¹`**. With this value the stimulus is far too weak to drive the system above the bifurcation and no decisions are reached. Wong & Wang (2006) — which Brady & Butler cite as their reference — state `J_ext = 5.2 × 10⁻⁴ nA·Hz⁻¹` (one order of magnitude larger). We use the W&W value; this is almost certainly a typo in B&B.

2. **SUM coupling type and weight**: the paper does not specify the equation for the SUM populations' synaptic current. We default to AMPA-style (`I_couple = J_couple_AMPA × (H_upstream)`) with a tunable weight, and also expose an NMDA-style option (`I_couple = J_couple_NMDA × (S_upstream)`).

3. **SUM module τ_S**: not given. We default to `τ_NMDA = 0.100 s`.

4. **REPEAT bump magnitude**: paper says "one σ of the noise" without specifying which variable. We expose both interpretations (see above).

## Noise options

Two noise models, applied to all populations (matching W&W 2006 / Brady & Butler):

- **`"ou"`** (default, B&B value): Ornstein–Uhlenbeck on the input current, filtered by `τ_AMPA = 2 ms` with σ = 0.007 nA. `dt`-independent.
- **`"gaussian"`**: independent Gaussian sample per timestep added to the current. Simple but `dt`-dependent.


## Setup: Drive and results folder (Colab) / local fallback

If running in Google Colab, mount Drive and use a folder there to save results. If running locally (e.g. with `jupyter notebook`), fall back to the current working directory. **Edit `tutorial_path` to point to your own folder before running.**


In [ ]:
import os

# If running in Google Colab, optionally point this to your own Drive folder.
# Otherwise leave as './' — the notebook will work in the current directory.
tutorial_path = './'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Optional: uncomment and edit the line below to save to your Drive instead
    # tutorial_path = '/content/drive/MyDrive/your_folder_here/'
    print(f"Running in Colab. Working in: {tutorial_path}")
except (ImportError, ModuleNotFoundError):
    tutorial_path = os.getcwd()
    print(f"Running locally. Working in: {tutorial_path}")

saved_results = os.path.join(tutorial_path, 'saved_results')
os.makedirs(saved_results, exist_ok=True)
print(f"Results will be saved under: {saved_results}")


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D


## Parameters

Slim dictionary of the parameters most likely to be varied. Hard-coded inside the simulation: the W&W f-I curve constants `(a, b, d)`, the kinetic parameter `γ = 0.641`, integration step `dt = 0.1 ms`, and `τ_AMPA = 2 ms` for the OU noise filter.

The connectivity weights `(J_HH, J_MM, J_HM, J_MH)` are the original W&W 2006 / B&B 2025 values for a single Hit/Miss module and are shared across all modules (audio, visual, SUM).

`τ_A`, `τ_V`, `τ_SUM` are the NMDA gating time constants of each module — Brady & Butler reinterpret these as per-participant, per-modality accumulation rates.

`sum_coupling_type` selects how the SUM module receives input from upstream:
- `'AMPA'`: rate-based, `I_couple_H = J_couple_AMPA × (H_AH + H_VH)` and similarly for the Miss pool.
- `'NMDA'`: S-based, `I_couple_H = J_couple_NMDA × (S_AH + S_VH)`.

`repeat_mode` selects how priming on a repeat trial is implemented:
- `'IC_bump'`: the initial S of the cued Hit population is set to `IC_bump` instead of 0.
- `'sigma_increase'`: the OU noise σ on the cued Hit population is multiplied by `sigma_factor` for that trial.


In [ ]:
PARAMS = {
    # --- Time grid (seconds) ---
    "time": {
        "dt": 1e-4,           # 0.1 ms (B&B value)
        "T_max": 2.0,         # max trial duration
    },

    # --- Connectivity (nA, single-module Wong-Wang values) ---
    "connectivity": {
        "J_HH": 0.2609,
        "J_MM": 0.2609,
        "J_HM": 0.0497,
        "J_MH": 0.0497,
        "I0":   0.3255,       # background current
    },

    # --- Stimulus encoding ---
    # NOTE: B&B state J_ext = 5.2e-5, but with that value the stimulus is
    # too weak to cross threshold. W&W 2006 (which B&B cite) state 5.2e-4.
    # Almost certainly a typo in B&B. We use 5.2e-4.
    "stimulus": {
        "J_ext":  5.2e-4,     # nA*Hz^-1 (W&W 2006 value)
        "mu0":    30.0,       # background firing of MT/upstream input (Hz)
        "c_stim": 12.0,       # stimulus strength (% — B&B value)
    },

    # --- Per-modality NMDA tau (B&B "accumulation rate") ---
    "tau": {
        "tau_A":   0.100,     # audio  module  (s)
        "tau_V":   0.100,     # visual module  (s)
        "tau_SUM": 0.100,     # SUM    module  (s)  -- our assumption
    },

    # --- SUM model coupling (paper does not specify; we expose both) ---
    "sum": {
        "sum_coupling_type": "AMPA",     # 'AMPA' or 'NMDA'
        "J_couple_AMPA":     0.002,      # nA*Hz^-1, for type='AMPA'
        "J_couple_NMDA":     0.10,       # nA,       for type='NMDA'
    },

    # --- REPEAT model priming ---
    "repeat": {
        "repeat_mode":   "IC_bump",      # 'IC_bump' or 'sigma_increase'
        "IC_bump":       0.05,
        "sigma_factor":  2.0,
    },

    # --- Noise ---
    "noise": {
        "noise_type":  "ou",     # 'ou' or 'gaussian'
        "sigma_noise": 0.007,    # nA  (B&B value)
        "tau_AMPA":    0.002,    # s   (OU filter time constant)
    },

    # --- Decision ---
    "decision": {
        "threshold": 20.0,        # Hz
    },

    # --- Simulation ---
    "simulation": {
        "n_trials": 10,
        "base_seed": 0,
    },
}


## Wong–Wang f-I curve

`H(x) = (a x − b) / (1 − exp(−d (a x − b)))` with `a = 270 V·nC⁻¹`, `b = 108 Hz`, `d = 0.154 s`. The L'Hôpital limit at the singularity `x = b/a` is `1/d`.


In [ ]:
def H_func(x, a=270.0, b=108.0, d=0.154):
    """Wong-Wang reduced f-I curve. Vectorized, handles 0/0 singularity."""
    num = a * x - b
    return np.where(
        np.abs(num) < 1e-6,
        1.0 / d,
        num / (1.0 - np.exp(-d * num) + 1e-30),
    )


## Core simulation: one trial

`simulate_trial(model_type, condition, PARAMS, prev_condition=None, seed=None)`
runs a single trial. Returns a dict with all rate trajectories, the reaction time `RT` (seconds, or `None` if no decision within `T_max`), the outcome (`'Hit'` / `'Miss'` / `None`), and which population crossed first.

The same dynamical equations (Wong-Wang) are evaluated for every module. The architecture differs only in (a) which populations get stimulus input, (b) whether the SUM module is present, and (c) the REPEAT priming on initial conditions or noise.


In [ ]:
def simulate_trial(model_type, condition, P, prev_condition=None, seed=None):
    """
    Single multisensory trial.

    Parameters
    ----------
    model_type : {'OR', 'SUM', 'REPEAT'}
    condition  : {'A', 'V', 'AV'}
    P          : PARAMS dict
    prev_condition : {'A', 'V', 'AV', None}  (only used by REPEAT)
    seed       : int or None
    """
    if seed is not None:
        np.random.seed(seed)

    # Unpack
    dt    = P["time"]["dt"]
    T_max = P["time"]["T_max"]
    JHH = P["connectivity"]["J_HH"]; JMM = P["connectivity"]["J_MM"]
    JHM = P["connectivity"]["J_HM"]; JMH = P["connectivity"]["J_MH"]
    I0  = P["connectivity"]["I0"]
    Jext = P["stimulus"]["J_ext"]; mu0 = P["stimulus"]["mu0"]; c = P["stimulus"]["c_stim"]
    tau_A = P["tau"]["tau_A"]; tau_V = P["tau"]["tau_V"]; tau_S = P["tau"]["tau_SUM"]
    gamma = 0.641
    a, b, d = 270.0, 108.0, 0.154

    sigma     = P["noise"]["sigma_noise"]
    tau_AMPA  = P["noise"]["tau_AMPA"]
    noise_type = P["noise"]["noise_type"]
    sqrt_dt_tau = np.sqrt(dt / tau_AMPA)        # OU
    sqrt_dt     = np.sqrt(dt)                   # gaussian

    threshold  = P["decision"]["threshold"]
    sum_type   = P["sum"]["sum_coupling_type"]
    Jc_AMPA    = P["sum"]["J_couple_AMPA"]
    Jc_NMDA    = P["sum"]["J_couple_NMDA"]
    repeat_mode = P["repeat"]["repeat_mode"]
    IC_bump     = P["repeat"]["IC_bump"]
    sigma_factor = P["repeat"]["sigma_factor"]

    n_steps = int(round(T_max / dt)) + 1
    t = np.arange(n_steps) * dt

    # External stimulus currents
    base = Jext * mu0
    I_AH_ext = base * (1 + c/100) if "A" in condition else 0.0
    I_AM_ext = base * (1 - c/100) if "A" in condition else 0.0
    I_VH_ext = base * (1 + c/100) if "V" in condition else 0.0
    I_VM_ext = base * (1 - c/100) if "V" in condition else 0.0

    # State arrays
    S_AH = np.zeros(n_steps); S_AM = np.zeros(n_steps)
    S_VH = np.zeros(n_steps); S_VM = np.zeros(n_steps)
    S_SH = np.zeros(n_steps); S_SM = np.zeros(n_steps)
    H_AH = np.zeros(n_steps); H_AM = np.zeros(n_steps)
    H_VH = np.zeros(n_steps); H_VM = np.zeros(n_steps)
    H_SH = np.zeros(n_steps); H_SM = np.zeros(n_steps)
    nAH = np.zeros(n_steps); nAM = np.zeros(n_steps)
    nVH = np.zeros(n_steps); nVM = np.zeros(n_steps)
    nSH = np.zeros(n_steps); nSM = np.zeros(n_steps)

    # Per-Hit-pop sigma (used by REPEAT 'sigma_increase' mode)
    sigma_AH = sigma; sigma_VH = sigma

    # REPEAT priming
    if model_type == "REPEAT" and prev_condition is not None:
        rep_A = ("A" in condition) and ("A" in prev_condition)
        rep_V = ("V" in condition) and ("V" in prev_condition)
        if repeat_mode == "IC_bump":
            if rep_A: S_AH[0] = IC_bump
            if rep_V: S_VH[0] = IC_bump
        elif repeat_mode == "sigma_increase":
            if rep_A: sigma_AH = sigma * sigma_factor
            if rep_V: sigma_VH = sigma * sigma_factor
        else:
            raise ValueError(f"Unknown repeat_mode: {repeat_mode}")

    use_SUM = (model_type == "SUM")

    def update_rates(n):
        x_AH = JHH*S_AH[n] - JHM*S_AM[n] + I0 + I_AH_ext + nAH[n]
        x_AM = JMM*S_AM[n] - JMH*S_AH[n] + I0 + I_AM_ext + nAM[n]
        x_VH = JHH*S_VH[n] - JHM*S_VM[n] + I0 + I_VH_ext + nVH[n]
        x_VM = JMM*S_VM[n] - JMH*S_VH[n] + I0 + I_VM_ext + nVM[n]
        H_AH[n] = H_func(x_AH, a, b, d); H_AM[n] = H_func(x_AM, a, b, d)
        H_VH[n] = H_func(x_VH, a, b, d); H_VM[n] = H_func(x_VM, a, b, d)
        if use_SUM:
            if sum_type == "AMPA":
                Ic_H = Jc_AMPA * (H_AH[n] + H_VH[n])
                Ic_M = Jc_AMPA * (H_AM[n] + H_VM[n])
            elif sum_type == "NMDA":
                Ic_H = Jc_NMDA * (S_AH[n] + S_VH[n])
                Ic_M = Jc_NMDA * (S_AM[n] + S_VM[n])
            else:
                raise ValueError(f"Unknown sum_coupling_type: {sum_type}")
            x_SH = JHH*S_SH[n] - JHM*S_SM[n] + I0 + Ic_H + nSH[n]
            x_SM = JMM*S_SM[n] - JMH*S_SH[n] + I0 + Ic_M + nSM[n]
            H_SH[n] = H_func(x_SH, a, b, d); H_SM[n] = H_func(x_SM, a, b, d)

    RT = None; outcome = None; decision_pop = None; decided = False

    for n in range(n_steps - 1):
        update_rates(n)

        # Noise update
        if noise_type == "ou":
            # OU: dI = -I/tau dt + sigma * sqrt(dt/tau) * xi
            nAH[n+1] = nAH[n] - dt*nAH[n]/tau_AMPA + sigma_AH*sqrt_dt_tau*np.random.randn()
            nAM[n+1] = nAM[n] - dt*nAM[n]/tau_AMPA + sigma   *sqrt_dt_tau*np.random.randn()
            nVH[n+1] = nVH[n] - dt*nVH[n]/tau_AMPA + sigma_VH*sqrt_dt_tau*np.random.randn()
            nVM[n+1] = nVM[n] - dt*nVM[n]/tau_AMPA + sigma   *sqrt_dt_tau*np.random.randn()
            if use_SUM:
                nSH[n+1] = nSH[n] - dt*nSH[n]/tau_AMPA + sigma*sqrt_dt_tau*np.random.randn()
                nSM[n+1] = nSM[n] - dt*nSM[n]/tau_AMPA + sigma*sqrt_dt_tau*np.random.randn()
        elif noise_type == "gaussian":
            nAH[n+1] = sigma_AH*sqrt_dt*np.random.randn()
            nAM[n+1] = sigma   *sqrt_dt*np.random.randn()
            nVH[n+1] = sigma_VH*sqrt_dt*np.random.randn()
            nVM[n+1] = sigma   *sqrt_dt*np.random.randn()
            if use_SUM:
                nSH[n+1] = sigma*sqrt_dt*np.random.randn()
                nSM[n+1] = sigma*sqrt_dt*np.random.randn()
        else:
            raise ValueError(f"Unknown noise_type: {noise_type}")

        # S update (Euler; dt = 0.1 ms is small enough)
        S_AH[n+1] = S_AH[n] + dt*(-S_AH[n]/tau_A + (1-S_AH[n])*gamma*H_AH[n])
        S_AM[n+1] = S_AM[n] + dt*(-S_AM[n]/tau_A + (1-S_AM[n])*gamma*H_AM[n])
        S_VH[n+1] = S_VH[n] + dt*(-S_VH[n]/tau_V + (1-S_VH[n])*gamma*H_VH[n])
        S_VM[n+1] = S_VM[n] + dt*(-S_VM[n]/tau_V + (1-S_VM[n])*gamma*H_VM[n])
        if use_SUM:
            S_SH[n+1] = S_SH[n] + dt*(-S_SH[n]/tau_S + (1-S_SH[n])*gamma*H_SH[n])
            S_SM[n+1] = S_SM[n] + dt*(-S_SM[n]/tau_S + (1-S_SM[n])*gamma*H_SM[n])

        # Decision check (on rates at step n)
        if not decided:
            if model_type in ("OR", "REPEAT"):
                if condition == "A":
                    pops = (("AH", H_AH[n]), ("AM", H_AM[n]))
                elif condition == "V":
                    pops = (("VH", H_VH[n]), ("VM", H_VM[n]))
                else:
                    pops = (("AH", H_AH[n]), ("AM", H_AM[n]),
                            ("VH", H_VH[n]), ("VM", H_VM[n]))
                for name, h in pops:
                    if h >= threshold:
                        RT = t[n]; decision_pop = name
                        outcome = "Hit" if name.endswith("H") else "Miss"
                        decided = True; break
            elif use_SUM:
                if H_SH[n] >= threshold:
                    RT = t[n]; decision_pop = "SH"; outcome = "Hit"; decided = True
                elif H_SM[n] >= threshold:
                    RT = t[n]; decision_pop = "SM"; outcome = "Miss"; decided = True

    update_rates(n_steps - 1)  # final-step rates for plotting

    return {
        "t": t,
        "H_AH": H_AH, "H_AM": H_AM, "H_VH": H_VH, "H_VM": H_VM,
        "H_SH": H_SH, "H_SM": H_SM,
        "RT": RT, "outcome": outcome, "decision_pop": decision_pop,
        "condition": condition, "model_type": model_type,
    }


def run_trials(model_type, condition, P, n_trials=None, prev_conditions=None,
               base_seed=None):
    """Run n_trials of one (model, condition). Returns list of trial dicts."""
    if n_trials is None: n_trials = P["simulation"]["n_trials"]
    if base_seed is None: base_seed = P["simulation"]["base_seed"]
    out = []
    for k in range(n_trials):
        prev = prev_conditions[k] if prev_conditions is not None else None
        out.append(simulate_trial(model_type, condition, P,
                                  prev_condition=prev, seed=base_seed + k))
    return out


## Plotting helpers

Three figures, in the style of B&B Figure 3:

- **Figure 1** — single example trial per condition (Audio, Visual, AV–OR, AV–SUM).
- **Figure 2** — overlay of `n_trials` AV trials, OR vs SUM side-by-side.
- **Figure 3** — REPEAT model: same RNG seed, repeat (`AV → AV`) vs switch (`V → AV`).


In [ ]:
HIT_COLORS  = {"A": "#1f4e8c", "V": "#a8202a", "S": "#404040"}
MISS_COLORS = {"A": "#7fa6d4", "V": "#e08a91", "S": "#a8a8a8"}
LABEL_MAP = {
    "AH": ("Audio Hit",  HIT_COLORS["A"]),
    "AM": ("Audio Miss", MISS_COLORS["A"]),
    "VH": ("Visual Hit", HIT_COLORS["V"]),
    "VM": ("Visual Miss", MISS_COLORS["V"]),
    "SH": ("SUM Hit",    HIT_COLORS["S"]),
    "SM": ("SUM Miss",   MISS_COLORS["S"]),
}


def plot_single_trial(ax, trial, show_pops=None, title=None, threshold=20.0):
    """Plot rate trajectories for one trial."""
    t_ms = trial["t"] * 1000
    if show_pops is None:
        cond = trial["condition"]; mt = trial["model_type"]
        if mt == "SUM":
            show_pops = ["SH", "SM"]
        elif cond == "A":
            show_pops = ["AH", "AM"]
        elif cond == "V":
            show_pops = ["VH", "VM"]
        else:
            show_pops = ["AH", "AM", "VH", "VM"]
    for pop in show_pops:
        label, color = LABEL_MAP[pop]
        ax.plot(t_ms, trial[f"H_{pop}"], color=color, label=label, lw=1.4)
    ax.axhline(threshold, color="k", ls="--", lw=0.8, alpha=0.6)
    if trial["RT"] is not None:
        ax.axvline(trial["RT"]*1000, color="green", ls=":", lw=1.0, alpha=0.7)
    ax.set_xlabel("Time (ms)"); ax.set_ylabel("Firing rate (Hz)")
    ax.set_ylim(-1, 35); ax.set_xlim(0, t_ms[-1])
    if title: ax.set_title(title, fontsize=10)
    ax.legend(loc="upper left", fontsize=8, frameon=False)


def figure_examples(P, savedir=None, seed=42):
    """Figure 1: one example trial per condition (A, V, AV-OR, AV-SUM)."""
    np.random.seed(seed)
    tr_A   = simulate_trial("OR",  "A",  P, seed=seed+1)
    tr_V   = simulate_trial("OR",  "V",  P, seed=seed+2)
    tr_OR  = simulate_trial("OR",  "AV", P, seed=seed+3)
    tr_SUM = simulate_trial("SUM", "AV", P, seed=seed+3)

    fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), sharey=True)
    def fmt(tr): return f"RT={tr['RT']*1000:.0f} ms  {tr['outcome']}" if tr['RT'] else "no decision"
    plot_single_trial(axes[0], tr_A,   title=f"Audio (OR)  {fmt(tr_A)}",   threshold=P["decision"]["threshold"])
    plot_single_trial(axes[1], tr_V,   title=f"Visual (OR) {fmt(tr_V)}",   threshold=P["decision"]["threshold"])
    plot_single_trial(axes[2], tr_OR,  title=f"AV — OR Model  {fmt(tr_OR)}",  threshold=P["decision"]["threshold"])
    plot_single_trial(axes[3], tr_SUM, title=f"AV — SUM Model ({P['sum']['sum_coupling_type']})  {fmt(tr_SUM)}", threshold=P["decision"]["threshold"])
    fig.suptitle(f"Example trials — τ_A=τ_V={P['tau']['tau_A']:.3f} s, c={P['stimulus']['c_stim']:.0f}%, threshold={P['decision']['threshold']:.0f} Hz", fontsize=11)
    fig.tight_layout()
    if savedir is not None:
        os.makedirs(savedir, exist_ok=True)
        fig.savefig(os.path.join(savedir, "fig1_example_trials.png"), dpi=130, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def figure_AV_overlay(P, savedir=None):
    """Figure 2: 10 AV trials overlaid for OR vs SUM."""
    n_trials = P["simulation"]["n_trials"]
    base_seed = P["simulation"]["base_seed"] + 100
    trials_OR  = run_trials("OR",  "AV", P, base_seed=base_seed)
    trials_SUM = run_trials("SUM", "AV", P, base_seed=base_seed)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
    threshold = P["decision"]["threshold"]
    Tmax_ms = P["time"]["T_max"] * 1000

    # OR panel
    for tr in trials_OR:
        t_ms = tr["t"]*1000
        axes[0].plot(t_ms, tr["H_AH"], color=HIT_COLORS["A"],  alpha=0.45, lw=0.9)
        axes[0].plot(t_ms, tr["H_AM"], color=MISS_COLORS["A"], alpha=0.45, lw=0.9)
        axes[0].plot(t_ms, tr["H_VH"], color=HIT_COLORS["V"],  alpha=0.45, lw=0.9)
        axes[0].plot(t_ms, tr["H_VM"], color=MISS_COLORS["V"], alpha=0.45, lw=0.9)
        if tr["RT"] is not None:
            axes[0].axvline(tr["RT"]*1000, color="green", ls=":", lw=0.5, alpha=0.4)
    axes[0].axhline(threshold, color="k", ls="--", lw=0.8)
    axes[0].set_title(f"AV — OR Model ({n_trials} trials)")
    axes[0].set_xlabel("Time (ms)"); axes[0].set_ylabel("Firing rate (Hz)")
    axes[0].set_ylim(-1, 35); axes[0].set_xlim(0, Tmax_ms)
    legend_OR = [Line2D([0],[0], color=HIT_COLORS["A"],  label="A_H"),
                 Line2D([0],[0], color=MISS_COLORS["A"], label="A_M"),
                 Line2D([0],[0], color=HIT_COLORS["V"],  label="V_H"),
                 Line2D([0],[0], color=MISS_COLORS["V"], label="V_M")]
    axes[0].legend(handles=legend_OR, loc="upper left", fontsize=8, frameon=False)

    # SUM panel
    for tr in trials_SUM:
        t_ms = tr["t"]*1000
        axes[1].plot(t_ms, tr["H_AH"], color=HIT_COLORS["A"], alpha=0.18, lw=0.7)
        axes[1].plot(t_ms, tr["H_VH"], color=HIT_COLORS["V"], alpha=0.18, lw=0.7)
        axes[1].plot(t_ms, tr["H_SH"], color=HIT_COLORS["S"],  alpha=0.55, lw=1.0)
        axes[1].plot(t_ms, tr["H_SM"], color=MISS_COLORS["S"], alpha=0.55, lw=1.0)
        if tr["RT"] is not None:
            axes[1].axvline(tr["RT"]*1000, color="green", ls=":", lw=0.5, alpha=0.4)
    axes[1].axhline(threshold, color="k", ls="--", lw=0.8)
    sum_type = P["sum"]["sum_coupling_type"]
    Jc = P["sum"]["J_couple_AMPA"] if sum_type=="AMPA" else P["sum"]["J_couple_NMDA"]
    axes[1].set_title(f"AV — SUM Model ({sum_type} coupling, J={Jc})")
    axes[1].set_xlabel("Time (ms)"); axes[1].set_xlim(0, Tmax_ms)
    legend_SUM = [Line2D([0],[0], color=HIT_COLORS["A"], alpha=0.5, label="A_H (upstream)"),
                  Line2D([0],[0], color=HIT_COLORS["V"], alpha=0.5, label="V_H (upstream)"),
                  Line2D([0],[0], color=HIT_COLORS["S"], label="SUM_H"),
                  Line2D([0],[0], color=MISS_COLORS["S"], label="SUM_M")]
    axes[1].legend(handles=legend_SUM, loc="upper left", fontsize=8, frameon=False)

    fig.suptitle(f"AV-condition: OR vs SUM ({n_trials} trials each)", fontsize=11)
    fig.tight_layout()
    if savedir is not None:
        os.makedirs(savedir, exist_ok=True)
        fig.savefig(os.path.join(savedir, "fig2_AV_overlay.png"), dpi=130, bbox_inches="tight")
    plt.show()
    plt.close(fig)

    # Print RT summary
    def _stats(trials):
        rts = [tr["RT"]*1000 for tr in trials if tr["RT"] is not None]
        hits = sum(1 for tr in trials if tr["outcome"]=="Hit")
        return rts, hits
    rts_or, hits_or = _stats(trials_OR)
    rts_sum, hits_sum = _stats(trials_SUM)
    print(f"OR  AV: mean RT = {np.mean(rts_or):6.1f} ms  ({len(rts_or)}/{n_trials} decided, {hits_or} Hits)")
    print(f"SUM AV: mean RT = {np.mean(rts_sum):6.1f} ms  ({len(rts_sum)}/{n_trials} decided, {hits_sum} Hits)")


def figure_repeat(P, savedir=None, seed=7):
    """Figure 3: REPEAT model — same seed, repeat vs switch trial."""
    tr_repeat = simulate_trial("REPEAT", "AV", P, prev_condition="AV", seed=seed)
    tr_switch = simulate_trial("REPEAT", "AV", P, prev_condition="V",  seed=seed)

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
    def fmt(tr): return f"RT={tr['RT']*1000:.0f} ms  {tr['outcome']}" if tr['RT'] else "no decision"
    plot_single_trial(axes[0], tr_switch,
                      title=f"AV after V — SWITCH    {fmt(tr_switch)}",
                      threshold=P["decision"]["threshold"])
    rep_mode = P["repeat"]["repeat_mode"]
    bump = P["repeat"]["IC_bump"] if rep_mode=="IC_bump" else f"σ×{P['repeat']['sigma_factor']}"
    plot_single_trial(axes[1], tr_repeat,
                      title=f"AV after AV — REPEAT (mode='{rep_mode}', {bump})    {fmt(tr_repeat)}",
                      threshold=P["decision"]["threshold"])
    fig.suptitle("REPEAT Model — same seed, different trial history", fontsize=11)
    fig.tight_layout()
    if savedir is not None:
        os.makedirs(savedir, exist_ok=True)
        fig.savefig(os.path.join(savedir, "fig3_repeat_vs_switch.png"), dpi=130, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def print_RT_summary(P):
    """Compact RT summary for OR and SUM, all 3 conditions, n_trials each."""
    n = P["simulation"]["n_trials"]
    base = P["simulation"]["base_seed"] + 200
    print(f"\nRT summary ({n} trials per cell)")
    print(f"  τ_A={P['tau']['tau_A']:.3f}s, τ_V={P['tau']['tau_V']:.3f}s, "
          f"sum_coupling={P['sum']['sum_coupling_type']}, noise={P['noise']['noise_type']}")
    for mt in ("OR", "SUM"):
        print(f"\n{mt} model:")
        for cond in ("A", "V", "AV"):
            trs = run_trials(mt, cond, P, base_seed=base)
            rts = [tr["RT"]*1000 for tr in trs if tr["RT"] is not None]
            hits = sum(1 for tr in trs if tr["outcome"]=="Hit")
            if rts:
                print(f"  {cond:3s}: mean RT = {np.mean(rts):6.1f} ± {np.std(rts):5.1f} ms  "
                      f"({len(rts)}/{n} decided, {hits} Hits)")
            else:
                print(f"  {cond:3s}: NO DECISION within T_max={P['time']['T_max']:.1f}s")


## Run

The cells below produce the three figures and print the RT summary using the parameters in `PARAMS` above. Edit `PARAMS` and re-run to explore.

### Things worth testing
- Change `PARAMS["sum"]["sum_coupling_type"]` to `"NMDA"` to see how rate-coupling vs S-coupling change SUM dynamics.
- Sweep `PARAMS["sum"]["J_couple_AMPA"]` (e.g. 0.0005, 0.001, 0.002, 0.004) to find a regime where SUM is faster than OR but not faster than the upstream Hit populations (the supra-optimal regime to be aware of).
- Change `PARAMS["repeat"]["repeat_mode"]` to `"sigma_increase"` and adjust `sigma_factor` to compare the two interpretations of "1 σ priming".
- Vary `PARAMS["tau"]["tau_A"]` and `PARAMS["tau"]["tau_V"]` (e.g. 0.080–0.130 s) to simulate participants with different unisensory accumulation rates.


In [ ]:
multisensory_savedir = os.path.join(saved_results, "multisensory_AV")
figure_examples(PARAMS, savedir=multisensory_savedir)


In [ ]:
figure_AV_overlay(PARAMS, savedir=multisensory_savedir)


In [ ]:
figure_repeat(PARAMS, savedir=multisensory_savedir)


In [ ]:
print_RT_summary(PARAMS)


## Biological context

### What the model represents

The three architectures here are **cognitive-level reductions** of audio-visual reaction-time tasks rather than detailed circuit models. Each `H/M` module is a Wong & Wang (2006) two-population decision unit, the same building block as the WTA notebook, but here interpreted as a **modality-specific evidence accumulator** for a detection task ("did a stimulus occur?") rather than a discrimination task ("which direction?"). The Hit population accumulates evidence *for* detection; the Miss population accumulates evidence *against*. The candidate brain substrates Brady & Butler invoke are the same LIP/PFC accumulator circuits that Wong & Wang (2006) and O'Connell et al. (2018) tie to perceptual decision-making, with the modality-specific modules standing in for parallel sensory-evidence streams that converge somewhere downstream.

The three architectures are *competing hypotheses* about what the convergence looks like — they are deliberately not refinements of a single biological story.

### OR Model — the race / Miller-bound baseline

The OR Model formalises a **non-integrative** strategy: the audio and visual modules run as independent accumulators, and the decision is whatever hits threshold first. This is the neural-circuit version of Raab's classic *race model* (Miller 1982): if two channels are independent, the probabilistic minimum of their finishing times is faster than either one alone, purely by chance. The bound Miller derived — that the AV cumulative distribution function cannot exceed the sum of the unisensory CDFs — is the standard yardstick for deciding whether observed AV speedup requires *integration* or whether independent racing suffices. The OR Model produces speedup that respects the Miller bound. Brady & Butler use this match to argue that the OR strategy plausibly describes populations whose multisensory benefits are within race-model predictions: young children (Brandwein et al. 2011) and children with autism (Brandwein et al. 2013, Foxe et al. 2015).

### SUM Model — coactivation in a downstream area

The SUM Model formalises **coactivation**: a third decision module receives convergent input from both unisensory modules and reaches threshold faster than either one could on its own. Brady & Butler motivate this with the Coen et al. (2023) result that mouse frontal cortex shows **additive multisensory neural activity** matching additive behavioural decisions on AV localisation — i.e., a downstream area whose activity is the linear sum of upstream sensory drives. The SUM Model is the mean-field analogue of that observation. Behaviourally, it produces speedup that *violates* the Miller bound (the hallmark of true integration; Miller 1982, Diederich 1995) and aligns with adult data and with populations showing strong AV benefits — older adults, people with Parkinson's (Fearon et al. 2015), and neurotypical adults.

A known limitation: with sufficiently strong coupling the SUM Model produces **supra-optimal** integration — AV reaction times faster than the statistically optimal Bayesian/quadratic prediction √(r_A² + r_V²) (Drugowitsch et al. 2014). This is not a desired feature; Brady & Butler note it as a falsifiable mismatch with behaviour and as a constraint on the coupling strength.

### REPEAT Model — separating priming from integration

The REPEAT Model addresses a confound first articulated by Shaw et al. (2020) and Crosse et al. (2022): in a mixed-modality block, AV trials are disproportionately preceded by single-modality trials (V or A), so the AV condition includes more *switch* trials than *repeat* trials. Switch trials carry a reaction-time cost (modality-switch cost; Gondan et al. 2004). If you do not control for this, the apparent AV benefit is partly a switch-cost artefact rather than genuine integration.

The REPEAT Model implements this by adding sensory priming to the OR architecture on repeat trials only — the cued Hit population gets a "baseline activity increase of one standard deviation of the noise" (Brady & Butler, citing Basso & Wurtz 1998 on prior-probability effects in superior colliculus and Huk & Shadlen 2005 on prior-effect signatures in LIP). Because the paper does not say which variable receives the bump, we expose both interpretations (`IC_bump` on S, `sigma_increase` on the noise σ). The point of the model is *negative*: showing that something that looks like AV integration can be reproduced by a non-integrative race + priming, so studies that ignore trial history may misattribute switch-cost-driven speedup to multisensory integration.

### Where this model differs from the others in this project

Unlike the WM, WTA, and CAI notebooks — which model neural circuits with reasonably tight anatomical anchoring — the multisensory model is **architectural and behavioural**. Each "module" is a population-level decision unit; the OR/SUM/REPEAT distinction is about *connectivity between modules*, not about the biology of one module. The single-module dynamics (Wong & Wang 2006: NMDA gating with τ_S ≈ 100 ms, the reduced f-I curve `H(x) = (ax − b) / (1 − exp(−d(ax − b)))`, OU current noise with τ_AMPA = 2 ms) carry the biological commitments. Everything above the module level — what couples to what, AMPA-vs-NMDA in the SUM coupling, the priming mechanism — is **hypothesised**, not measured. Brady & Butler's contribution is to make these hypotheses simulatable and falsifiable against AV reaction-time distributions.

### The τ_S reinterpretation

Wong & Wang (2006) treat τ_S = 100 ms as a fixed biophysical NMDA-channel time constant. Brady & Butler **re-purpose** it as a *per-participant sensitivity parameter* (faster τ_S → faster integration), sweeping τ_A, τ_V ∈ [0.080, 0.130] s across 121 simulated participants to capture inter-individual variability in audio-vs-visual dominance. This is a substantive departure from the original biophysical interpretation: in W&W, τ_S is fixed by NMDA kinetics; in B&B, it is the parameter that lets one participant be "audio-fast" and another "visual-fast." Whether differences in NMDA channel kinetics actually underlie inter-individual differences in modality dominance is an open empirical question — what is clear is that *something* differs across participants, and B&B map it onto τ_S as a tractable knob.

### What this model can and cannot tell you

It *can* tell you:
- Whether observed AV speedup is consistent with race-model (independent channels) or requires coactivation (linear summation)
- How modality-repeat priming can mimic integration in mixed designs, and how to design experiments that disentangle them
- How per-participant differences in accumulation rate translate into the distribution of unisensory and multisensory reaction times across a population
- That the same downstream decision module (Wong–Wang) can express very different multisensory behaviours depending purely on its upstream wiring

It *cannot* tell you:
- Where in the brain the SUM area lives (mouse frontal cortex per Coen et al. 2023 is the closest anatomical anchor B&B cite, but the model is agnostic)
- Whether the SUM coupling is AMPA-mediated, NMDA-mediated, or both — the paper does not specify and we leave it as a parameter
- How sensory weights are inferred (no Bayesian/reliability-weighting machinery; for that see Alais & Burr 2019, Körding et al. 2007)
- Anything about congruent-vs-incongruent or spatially-disparate AV stimuli — B&B model congruent salient stimuli only
- The neural substrate of the modality-switch cost itself (it is encoded here as a phenomenological priming bump, not derived)